In [1]:
from pydantic import BaseModel,field_validator,EmailStr,model_validator
from typing import Optional,Any
from enum import Enum
from datetime import datetime

### 1. Basic Classes

In [ ]:
class User(BaseModel):
    name: str
    age:int
    contact: int
    city: str
    country: str

: 

In [ ]:
user = User(name="Abhishek",age = 26, contact=9876110234,
            city="New Delhi", country= "India")

: 

In [ ]:
print(user)

: 

##### 1.2. with Optional Parameter and default values

In [ ]:
class Animal(BaseModel):
    species: str
    habitat: str
    diet: str
    weight: Optional[int]
    age: Optional[int]
    is_pet: bool

: 

In [ ]:
tiger = Animal(species="Tiger",habitat="Jungle", 
               diet= "Carnivours", weight = 120,age = 15, is_pet=False)

: 

In [ ]:
cat = Animal(species="cat",habitat="City", 
               diet= "Omnivours", weight = 25, is_pet=False)

: 

In [ ]:
print("""Learning = Surrounding your variable with only Optional in type hint is 
      not enough you need to add =None {age: Optional[int]=None} for the 
      value to be actually optional or you can also use the new style 
      {age: int | None=None}.
      If you notice even after providing pipe operator or Optional we 
      had to pass {=None} because having Optional and pipe operator tells 
      the code the value can be an int or None but is still required. 
      so passing a default value as =None lets it pass the data and 
      actually be none [Try passing value =None in data model creation 
      without using pipe or Optional in declaration]""")


: 

In [ ]:
class Animal(BaseModel):
    species: str
    habitat: str
    diet: str
    weight: Optional[int]
    age: int
    is_pet: bool

: 

In [ ]:
cat = Animal(species="cat",habitat="City", 
               diet= "Omnivours", weight = 25,age=None, is_pet=False)

: 

In [ ]:
# Correct Class Declaration
class Animal(BaseModel):
    species: str
    habitat: str
    diet: str
    weight: Optional[int]
    age: int | None = None
    is_pet: bool

: 

In [ ]:
cat = Animal(species="cat",habitat="City", 
               diet= "Omnivours", weight = 25, is_pet=False) #age=None

: 

### 2. Nested Classes

In [ ]:
class Address(BaseModel):
    line_1:str
    city:str
    country:str
    pin_code:int
    
class User(BaseModel):
    name:str
    contact:int
    email:str
    address:Address

: 

In [ ]:
test_user = User(name="Abhishek",contact="9643911223",email="abhishekbh44@hotmail.com",address=Address(line_1="Sadh Nagar",city="New Delhi",country="India",pin_code=110043))

: 

In [ ]:
print(test_user.name)

: 

In [ ]:
print(test_user.address)

: 

In [ ]:
print(test_user.address.line_1)

: 

#### Advanced Nested Classes

In [2]:
class Diet(str, Enum):
    herbivore = "herbivore"
    carnivore = "carnivore"
    omnivore = "omnivore"

    
class Animal(BaseModel):
    name: str
    species: str
    age: int | None =None
    weight: int | None = None
    diet: Diet
    is_endangered:bool = False

class Veterenian(BaseModel):
    name:str
    experience_years: int
    specialization: list[str]
    available:bool = True
                 
class Enclosure(BaseModel):
    enclosure_id: int
    habitat_type: str
    animals: list[Animal]
    assigned_vet: Veterenian
    capacity:int

class Zoo(BaseModel):
    name: str
    city: str
    enclosures: list[Enclosure]
    established_year: int
    open_to_public: bool = True

In [3]:
zoo = Zoo(
    name="National Zoo",
    city="Bangalore",
    established_year=1995,
    enclosures=[
        {
            "enclosure_id": 1,
            "habitat_type": "Savannah",
            "capacity": 10,
            "assigned_vet": {
                "name": "Dr. Sharma",
                "experience_years": 12,
                "specialization": ["Mammals", "Wildlife"]
            },
            "animals": [
                {
                    "name": "Leo",
                    "species": "Lion",
                    "diet": "carnivore",
                    "age": 5
                }
            ]
        }
    ]
)

##### With Field Validators

In [ ]:
class Animal(BaseModel):
    name: str
    species: str
    age: int
    weight: int
    diet: Diet
    is_endangered:bool = False

    # @field_validator("age")
    # def validate_age(cls,v):
    #     if v<=0:
    #         raise ValueError("The animal should be born before getting registered😊")
        
    # @field_validator("weight")
    # def validate_weight(cls,v):
    #     if v<=0:
    #         raise ValueError("Animal Weight can not be negative")
   
    @field_validator("age", "weight")
    def validate_positive(cls, v):
        if v < 0:
            raise ValueError("Must be positive")
        return v
class Enclosure(BaseModel):
    enclosure_id:int
    habitat_type: str
    capacity:int
    assigned_vet:Veterenian
    animals: list[Animal]

    @field_validator("capacity")
    def neg_capacity(cls,v):
        if v<=0:
            raise ValueError("Zoo Capacity shouldbe higher than 0")

    


class zoo(BaseModel):
    name: str
    city: str
    enclosures:list[Enclosure]
    established_year: int
    open_to_public: bool =True

    @field_validator("established_year")
    def validate_year(cls,v):
        if v>datetime.now().year:
            raise ValueError("The Establishment Date can't be of future")

In [5]:
zootopia = zoo(name="Zotopia",city="Unknown",enclosures=[
        {
            "enclosure_id": 1,
            "habitat_type": "Savannah",
            "capacity": 10,
            "assigned_vet": {
                "name": "Dr. Sharma",
                "experience_years": 12,
                "specialization": ["Mammals", "Wildlife"]
            },
            "animals": [
                {
                    "name": "Leo",
                    "species": "Lion",
                    "diet": "carnivore",
                    "age": 5,
                    "weight":10 #-10,0
                }
            ]
        }
    ],
    established_year=2025,
    open_to_public=True)

##### with model validator

In [ ]:
class Credentials(BaseModel):
    password: Any
    confirm_password: Any
    @model_validator(mode='after')
    def password_match(self):
        if self.password!=self.confirm_password:
            raise ConnectionError("Password and Confirmed Password don't match")
        return self

class User(BaseModel):
    name: str
    contact: str
    email:EmailStr
    credentials: Credentials

    @field_validator("contact")
    def validate_phone(cls,v):
        if not(10==len(v) or len(v)==12):  # Understand the difference between |(Bitwise OR: Used for Binary Operations) and or (Logical OR: Used for Logical Conditions)
            raise ValueError(f"The Phone Number should be of 10 digits or 12 with country code. Current digits length: {len(v)}.")
        elif len(v)==12:
            if not v.startswith(tuple(["91","01"])): #.startswith() only accepts tuple and not lists
                raise ValueError("Not a Valid Country Code")
        return v
    

In [37]:
user_test = User(name="Atul",contact="919871002130",email="atuljha@gmail.com",credentials=Credentials(password="22113",confirm_password="22113"))